# 01 — Check and prepare Liu's original gene table

This notebook opens the original spreadsheet published with Liu et al. (2014) and pulls out the 369 genes in the study's secretion-machinery list. It checks that the gene IDs and expression results are complete, identifies the 51 genes that changed consistently in all three high-secretion strains, and saves a clean copy for the next notebook. The separate list of proteins predicted to be secreted is not included in this project.

Set up the project paths and confirm that the original Liu workbook is available before reading it.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from atlas.schema import SUBSYSTEM_ORDER

RAW_LIU_DIR = Path("../data/raw/liu2014")
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
WORKBOOK = RAW_LIU_DIR / "12918_2013_1339_MOESM2_ESM.xls"
assert WORKBOOK.exists(), f"Missing Liu workbook: {WORKBOOK}"

### Inspect the source workbook

List every worksheet and its size so the source workbook can be checked before selecting the machinery table.

In [2]:
excel = pd.ExcelFile(WORKBOOK)
print(WORKBOOK.name, WORKBOOK.stat().st_size, "bytes")
print("sheets:", excel.sheet_names)

profile = {"source_file": WORKBOOK.name, "format": "xls", "sheets": {}}
for sheet in excel.sheet_names:
    raw = pd.read_excel(WORKBOOK, sheet_name=sheet, header=None)
    profile["sheets"][sheet] = {"physical_rows": len(raw), "physical_columns": raw.shape[1]}
    print(f"{sheet}: {raw.shape[0]} physical rows x {raw.shape[1]} columns")

12918_2013_1339_MOESM2_ESM.xls 3927040 bytes
sheets: ['Table S1', 'Table S2', 'Table S3', 'Table S4', 'Table S5']


Table S1: 371 physical rows x 14 columns


Table S2: 12898 physical rows x 11 columns


Table S3: 2271 physical rows x 9 columns


Table S4: 1020 physical rows x 10 columns


Table S5: 1029 physical rows x 6 columns


#### Load the 369-gene worksheet

Read Table S1 with the correct header row and confirm that all 369 machinery records are present.

In [3]:
# Row 1 is a title; row 2 contains the real headers in Table S1.
components = pd.read_excel(WORKBOOK, sheet_name="Table S1", header=1)
component_source_columns = components.columns.tolist()
assert len(components) == 369
profile["sheets"]["Table S1"].update({"data_rows": len(components), "columns": component_source_columns})
print("machinery components:", len(components), component_source_columns)

machinery components: 369 ['ID', 'S. cerevisiae ortholog', 'Subsystems or function', 'Description', 'CF1.1 vs A1560_logFC', 'A16 vs A1560_logFC', 'CF32 vs A1560_logFC', 'CF1.1vs A1560_adj.P.Val', 'A16 vs A1560_adj.P.Val', 'CF32 vs A1560_adj.P.Val', '1st SOURCE', '2nd SOURCE', '3rd SOURCE', '4th SOURCE']


### Check for missing or repeated gene IDs

Check that the gene IDs have the expected format and identify missing IDs, duplicate rows, and cases where several *A. oryzae* genes share one yeast counterpart.

In [4]:
AO_PATTERN = r"AO090\d{9}"
gene_ids = components["ID"]
machinery_stats = {
    "rows": len(components),
    "missing_ids": int(gene_ids.isna().sum()),
    "duplicate_id_rows": int(components.duplicated("ID", keep=False).sum()),
    "unique_ids": int(gene_ids.nunique(dropna=True)),
    "ao090_format_matches": int(gene_ids.astype(str).str.fullmatch(AO_PATTERN).sum()),
}
profile["machinery"] = machinery_stats
print("machinery", machinery_stats)

per_yeast = components.groupby("S. cerevisiae ortholog")["ID"].nunique()
profile["machinery"]["yeast_orthologs_with_multiple_ao_genes"] = int((per_yeast > 1).sum())
print("yeast orthologs mapping to >1 A. oryzae ID:", int((per_yeast > 1).sum()))

machinery {'rows': 369, 'missing_ids': 0, 'duplicate_id_rows': 0, 'unique_ids': 369, 'ao090_format_matches': 369}
yeast orthologs mapping to >1 A. oryzae ID: 6


### Standardize the pathway categories

Standardize Liu's pathway labels and check that each one fits the atlas's allowed subsystem names.

In [5]:
SUBSYSTEM_NORMALIZATION = {
    "TC": "tc", "Dolichol pathway": "dolichol_pathway",
    "Erglycosylation": "er_glycosylation", "Folding": "folding",
    "GPI biosynthesis": "gpi_biosynthesis", "ERAD": "erad",
    "COPII": "copii", "COPI": "copi",
    "Golgi processing": "golgi_processing", "LDSV": "ldsv",
    "HDSV": "hdsv", "CPY pathway": "cpy_pathway",
    "ALPpathway": "alp_pathway", "SNARE": "snare",
    "Septin": "septin",
    "beta-1,6 glucan biosynthesis": "beta_1_6_glucan_biosynthesis",
    "Translation": "translation",
    "putative mitochondria protein": "putative_mitochondria_protein",
    "mitochondrial m‐AAA protease": "mitochondrial_m_aaa_protease",
}
observed_raw = sorted(components["Subsystems or function"].dropna().unique())
observed_normalized = {SUBSYSTEM_NORMALIZATION[x] for x in observed_raw}
schema_subsystems = set(SUBSYSTEM_ORDER)
profile["subsystems"] = {
    "raw_labels": observed_raw,
    "uncategorized_rows": int(components["Subsystems or function"].isna().sum()),
    "observed_not_in_schema": sorted(observed_normalized - schema_subsystems),
    "schema_not_observed": sorted(schema_subsystems - observed_normalized),
}
print(json.dumps(profile["subsystems"], indent=2, ensure_ascii=False))

{
  "raw_labels": [
    "ALPpathway",
    "COPI",
    "COPII",
    "CPY pathway",
    "Dolichol pathway",
    "ERAD",
    "Erglycosylation",
    "Folding",
    "GPI biosynthesis",
    "Golgi processing",
    "HDSV",
    "LDSV",
    "SNARE",
    "Septin",
    "TC",
    "Translation",
    "beta-1,6 glucan biosynthesis",
    "mitochondrial m‐AAA protease",
    "putative mitochondria protein"
  ],
  "uncategorized_rows": 260,
  "observed_not_in_schema": [],
  "schema_not_observed": []
}


### Check the three expression comparisons

The paper defines differential expression as adjusted p-value < 0.05. `sig_all_three` therefore requires all three exact adjusted-p columns to be below 0.05. Direction is assigned only when all three corresponding logFC values have the same sign.

In [6]:
LOGFC_COLUMNS = [
    "CF1.1 vs A1560_logFC", "A16 vs A1560_logFC", "CF32 vs A1560_logFC",
]
ADJ_P_COLUMNS = [
    "CF1.1vs A1560_adj.P.Val", "A16 vs A1560_adj.P.Val",
    "CF32 vs A1560_adj.P.Val",
]
SIGNIFICANCE_THRESHOLD = 0.05
assert all(c in components.columns for c in LOGFC_COLUMNS + ADJ_P_COLUMNS)
assert all(pd.api.types.is_numeric_dtype(components[c]) for c in LOGFC_COLUMNS + ADJ_P_COLUMNS)
profile["transcriptomics"] = {
    "logfc_columns": LOGFC_COLUMNS, "adjusted_p_columns": ADJ_P_COLUMNS,
    "criterion": "all three adjusted p-values < 0.05",
    "threshold": SIGNIFICANCE_THRESHOLD,
}
print(json.dumps(profile["transcriptomics"], indent=2))

{
  "logfc_columns": [
    "CF1.1 vs A1560_logFC",
    "A16 vs A1560_logFC",
    "CF32 vs A1560_logFC"
  ],
  "adjusted_p_columns": [
    "CF1.1vs A1560_adj.P.Val",
    "A16 vs A1560_adj.P.Val",
    "CF32 vs A1560_adj.P.Val"
  ],
  "criterion": "all three adjusted p-values < 0.05",
  "threshold": 0.05
}


#### Find genes that changed consistently

Mark genes that are significant in all three comparisons and record a direction only when all three changes agree. The expected result is 51 genes: 48 increased and 3 decreased.

In [7]:
# Build the two summary fields only after checking the source columns above.
def add_all_three_flags(frame):
    out = frame.copy()
    out["sig_all_three"] = out[ADJ_P_COLUMNS].lt(SIGNIFICANCE_THRESHOLD).all(axis=1)
    all_up = out[LOGFC_COLUMNS].gt(0).all(axis=1)
    all_down = out[LOGFC_COLUMNS].lt(0).all(axis=1)
    out["direction_all_three"] = None
    out.loc[out["sig_all_three"] & all_up, "direction_all_three"] = "up"
    out.loc[out["sig_all_three"] & all_down, "direction_all_three"] = "down"
    return out

components = add_all_three_flags(components)
machinery_counts = components.loc[components.sig_all_three, "direction_all_three"].value_counts().to_dict()
assert int(components.sig_all_three.sum()) == 51
assert machinery_counts == {"up": 48, "down": 3}
profile["derived_counts"] = {
    "machinery_sig_all_three": int(components.sig_all_three.sum()),
    "machinery_direction_all_three": machinery_counts,
}
print(profile["derived_counts"])

{'machinery_sig_all_three': 51, 'machinery_direction_all_three': {'up': 48, 'down': 3}}


### Save the cleaned source data

Run final checks that no source rows or columns were lost, then save the cleaned source table and a short report of any parsing issues.

In [8]:
issues = [
    {"scope": "workbook", "issue": "Table S1 uses a title row above the real header; parsed with header=1"},
]
missing_tx = int(components[LOGFC_COLUMNS + ADJ_P_COLUMNS].isna().any(axis=1).sum())
if missing_tx:
    issues.append({"scope": "machinery", "issue": f"{missing_tx} rows have at least one missing transcriptomic value"})

assert len(components) == 369
assert components.columns[:len(component_source_columns)].tolist() == component_source_columns
assert set(LOGFC_COLUMNS + ADJ_P_COLUMNS).issubset(components.columns)

components.to_csv(INTERIM_DIR / "liu_components_raw_cleaned.csv", index=False)
pd.DataFrame(issues).to_csv(INTERIM_DIR / "parsing_issues.csv", index=False)
profile["parsing_issues"] = issues
with open(INTERIM_DIR / "source_profile.json", "w", encoding="utf-8") as handle:
    json.dump(profile, handle, indent=2, ensure_ascii=False)
print("saved machinery source file; all Table S1 rows and columns retained")

saved machinery source file; all Table S1 rows and columns retained

### What this notebook confirms

- Table S1 machinery source row counts and original columns are asserted before export.
- Exact transcriptomic columns and p < 0.05 criterion are recorded before aggregate flags are calculated.
- The machinery result reproduces Liu's 51 total (48 up, 3 down).
- Liu's predicted secretome remains available in supplementary Table S3 and is out of scope here.